# Headline sentiment

Scores every WNBA headline with a RoBERTa sentiment model and regenerates two CSVs in `src/data/`:
`headline_sentiment.csv` (every Fox headline, scored −1…+1) and `sentiment_outlets.csv`
(per-player mean sentiment at each outlet, so Fox has a comparison).

Needs `pip install torch transformers` and takes ~10 minutes on an Apple Silicon Mac
(the model scores ~50,000 headlines). One caveat baked into the data: ESPN, CBS and USA Today
headlines are reconstructed from URL slugs (lowercase, no punctuation), which shifts the model
slightly — like-for-like comparisons should prefer Fox vs. Yahoo, which are both real headlines.


In [ ]:
import json, re
from collections import defaultdict
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DATA = Path("../data")
OUT = Path("../src/data")

MODEL = "cardiffnlp/twitter-roberta-large-topic-sentiment-latest"
BATCH = 64
# expected value across the model's ordinal classes
LABEL_WEIGHTS = [("strongly negative", -1.0), ("strongly positive", 1.0),
                 ("negative or neutral", 0.0), ("positive or neutral", 0.0),
                 ("negative", -0.6), ("positive", 0.6), ("neutral", 0.0)]
BUCKET_EDGE = 0.15

def weight_for(label):
    return next((w for phrase, w in LABEL_WEIGHTS if phrase in label), 0.0)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)
device = "mps" if torch.backends.mps.is_available() else "cpu"
model.to(device).eval()
weights = [weight_for(model.config.id2label[i].lower()) for i in range(model.config.num_labels)]

def score(texts):
    out = []
    for start in range(0, len(texts), BATCH):
        chunk = texts[start:start + BATCH]
        encoded = tokenizer(chunk, padding=True, truncation=True, max_length=64,
                            return_tensors="pt").to(device)
        with torch.no_grad():
            probs = torch.softmax(model(**encoded).logits, dim=-1).cpu().tolist()
        out += [round(sum(p * w for p, w in zip(prob, weights)), 4) for prob in probs]
        if (start // BATCH) % 40 == 0:
            print(f"  {start + len(chunk):,}/{len(texts):,}", flush=True)
    return out

print(f"model on {device}")

## Every Fox headline


In [ ]:
fox = pd.read_csv(DATA / "fox_wnba.csv")
fox_headlines = list(dict.fromkeys(h.strip() for h in fox.headline.dropna() if h.strip()))
fox_scores = score(fox_headlines)

def bucket(s):
    return "negative" if s <= -BUCKET_EDGE else "positive" if s >= BUCKET_EDGE else "neutral"

sent = (pd.DataFrame({"headline": fox_headlines, "score": fox_scores})
          .assign(label=lambda d: d.score.map(bucket))
          .sort_values("score"))

#### Overwrites `src/data/headline_sentiment.csv`

In [ ]:
sent.to_csv(OUT / "headline_sentiment.csv", index=False)
print(sent.label.value_counts().to_dict())
sent.head(5)

## Every outlet, per player


In [ ]:
index = pd.read_csv(DATA / "all_outlets_index.csv")
others = (index[~index.outlet.str.startswith("Fox")]
          .assign(headline=lambda d: d.headline.str.strip())
          .dropna(subset=["headline"])
          .drop_duplicates(["outlet", "headline"]))
other_scores = dict(zip(zip(others.outlet, others.headline), score(list(others.headline))))

players = {n["name"] for n in json.loads((OUT / "headline_network.json").read_text())["nodes"]
           if n["group"] == "players"}
name_rx = {n: re.compile(r"\b" + re.escape(n.split()[-1]) + r"\b", re.I) for n in players}

pairs = defaultdict(list)
for (outlet, headline), s in other_scores.items():
    for name, rx in name_rx.items():
        if rx.search(headline):
            pairs[(name, outlet)].append(s)
for headline, s in zip(fox_headlines, fox_scores):
    for name, rx in name_rx.items():
        if rx.search(headline):
            pairs[(name, "Fox News")].append(s)

rows = [{"player": name, "outlet": outlet, "n": len(v), "mean": round(sum(v) / len(v), 4)}
        for (name, outlet), v in sorted(pairs.items()) if len(v) >= 12]

#### Overwrites `src/data/sentiment_outlets.csv`

In [ ]:
pd.DataFrame(rows).to_csv(OUT / "sentiment_outlets.csv", index=False)
pd.DataFrame(rows).query("outlet in ('Fox News', 'Yahoo Sports')").sort_values(["player", "outlet"]).head(12)